In [1]:
!pip install polars -q

In [2]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

import gc
import numpy as np
import polars as pl
import pandas as pd
import joblib

In [3]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [4]:
try:
    import pyarrow
    print("PyArrow is installed. Version:", pyarrow.__version__)
except ModuleNotFoundError:
    print("PyArrow is NOT found in this environment. You installed it in a different one.")

PyArrow is installed. Version: 23.0.1


In [5]:
train = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1600/train_combined_1600.parquet')

In [6]:
test = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1600/test_combined_1600.parquet')

In [7]:
target = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_target.parquet')

In [8]:
X_train_pd = train.drop('customer_id').to_pandas().astype('float32')

In [9]:
y_train_pd = target.drop('customer_id').to_pandas()

In [10]:
del train
gc.collect()

0

In [11]:
hgb_base = HistGradientBoostingClassifier(
    max_iter=500,               
    max_depth=6,
    learning_rate=0.05,           
    min_samples_leaf=20,        
    l2_regularization=3.0,       
    max_bins=64,
    categorical_features=None,    
    random_state=42,
    verbose=0,                     
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
)

multi_hgb = MultiOutputClassifier(hgb_base)
multi_hgb.fit(X_train_pd, y_train_pd)

MultiOutputClassifier(estimator=HistGradientBoostingClassifier(categorical_features=None,
                                                               early_stopping=True,
                                                               l2_regularization=3.0,
                                                               learning_rate=0.05,
                                                               max_bins=64,
                                                               max_depth=6,
                                                               max_iter=500,
                                                               random_state=42))

In [12]:
sample_submit = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/sample_submit.parquet')
raw_preds = []

for est in multi_hgb.estimators_:
    raw_preds.append(est.decision_function(test.drop('customer_id').to_numpy().astype('float32')))

test_raw = np.column_stack(raw_preds)

predict_schema = [col.replace("target_", "predict_") for col in target.columns if col.startswith("target_")]

hgb_submit = pl.DataFrame(
    test_raw,
    schema=predict_schema,
    orient='row'
)

hgb_submit = pl.concat([test.select('customer_id'), hgb_submit], how='horizontal')

hgb_submit.write_parquet("hgb_submit_1600.parquet")